# Modulo 1: Ingesta y Muestreo Temporal

Este cuaderno recorre el dataset, extrae metadatos por video y genera los manifests temporales base.


## Librerias y parametros de ingesta


In [1]:
from pathlib import Path
import json
import math

import cv2
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "module1"

TARGET_FPS = 5
WINDOW_SECONDS = 4
STRIDE_SECONDS = 2
FRAMES_PER_WINDOW = 8

SPLITS = ["train", "val", "test"]
CLASS_NAMES = ["normal", "hurto_simulado"]

pd.Series(
    {
        "DATA_DIR": str(DATA_DIR),
        "OUTPUT_DIR": str(OUTPUT_DIR),
        "TARGET_FPS": TARGET_FPS,
        "WINDOW_SECONDS": WINDOW_SECONDS,
        "STRIDE_SECONDS": STRIDE_SECONDS,
        "FRAMES_PER_WINDOW": FRAMES_PER_WINDOW,
    }
)


DATA_DIR             c:\Users\franco\Downloads\1-INTELIGENCIA_ARTIF...
OUTPUT_DIR           c:\Users\franco\Downloads\1-INTELIGENCIA_ARTIF...
TARGET_FPS                                                           5
WINDOW_SECONDS                                                       4
STRIDE_SECONDS                                                       2
FRAMES_PER_WINDOW                                                    8
dtype: object

## Videos encontrados en el dataset


In [2]:
video_rows = []

for split_name in SPLITS:
    for class_name in CLASS_NAMES:
        class_dir = DATA_DIR / split_name / class_name
        for video_path in sorted(class_dir.glob("*.mp4")):
            video_rows.append(
                {
                    "split": split_name,
                    "class_name": class_name,
                    "video_path_absolute": str(video_path.resolve()),
                    "video_path": video_path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix(),
                }
            )

video_items_df = pd.DataFrame(video_rows).sort_values(["split", "class_name", "video_path"]).reset_index(drop=True)

print(f"Videos descubiertos: {len(video_items_df)}")
display(video_items_df.groupby(["split", "class_name"]).size().reset_index(name="num_videos"))
display(video_items_df.head())


Videos descubiertos: 328


,split,class_name,num_videos
0,test,hurto_simulado,17
1,test,normal,33
2,train,hurto_simulado,76
3,train,normal,153
4,val,hurto_simulado,16
5,val,normal,33


,split,class_name,video_path_absolute,video_path
0,test,hurto_simulado,C:\Users\franco\Downloads\1-INTELIGENCIA_ARTIF...,data/test/hurto_simulado/shop_lifter_102.mp4
1,test,hurto_simulado,C:\Users\franco\Downloads\1-INTELIGENCIA_ARTIF...,data/test/hurto_simulado/shop_lifter_17.mp4
2,test,hurto_simulado,C:\Users\franco\Downloads\1-INTELIGENCIA_ARTIF...,data/test/hurto_simulado/shop_lifter_30.mp4
3,test,hurto_simulado,C:\Users\franco\Downloads\1-INTELIGENCIA_ARTIF...,data/test/hurto_simulado/shop_lifter_38.mp4
4,test,hurto_simulado,C:\Users\franco\Downloads\1-INTELIGENCIA_ARTIF...,data/test/hurto_simulado/shop_lifter_40.mp4


## Metadatos por video


In [3]:
video_metadata_rows = []

for _, row in video_items_df.iterrows():
    split_name = str(row["split"])
    class_name = str(row["class_name"])
    video_path_absolute = Path(str(row["video_path_absolute"]))

    capture = cv2.VideoCapture(str(video_path_absolute))
    if not capture.isOpened():
        raise RuntimeError(f"No se pudo abrir el video: {video_path_absolute}")

    fps_original = float(capture.get(cv2.CAP_PROP_FPS))
    frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    capture.release()

    if fps_original <= 0:
        raise ValueError(f"FPS invalido para el video: {video_path_absolute}")

    duration_seconds = frame_count / fps_original
    video_metadata_rows.append(
        {
            "split": split_name,
            "class_name": class_name,
            "video_path": str(row["video_path"]),
            "video_id": f"{split_name}__{class_name}__{video_path_absolute.stem}",
            "fps_original": round(fps_original, 6),
            "frame_count": int(frame_count),
            "duration_seconds": round(duration_seconds, 6),
            "width": int(width),
            "height": int(height),
        }
    )

video_df = pd.DataFrame(video_metadata_rows).sort_values(["split", "class_name", "video_id"]).reset_index(drop=True)

print(f"Videos con metadatos: {len(video_df)}")
display(
    video_df.groupby(["split", "class_name"])
    .agg(
        num_videos=("video_id", "size"),
        duracion_media_s=("duration_seconds", "mean"),
        frames_promedio=("frame_count", "mean"),
    )
    .round(2)
    .reset_index()
)


Videos con metadatos: 328


,split,class_name,num_videos,duracion_media_s,frames_promedio
0,test,hurto_simulado,17,13.42,332.88
1,test,normal,33,16.48,412.12
2,train,hurto_simulado,76,13.85,343.80
3,train,normal,153,13.68,341.99
4,val,hurto_simulado,16,13.70,340.06
5,val,normal,33,13.70,342.42


## Ventanas temporales por video


In [4]:
window_rows = []

for _, video_row in video_df.iterrows():
    split_name = str(video_row["split"])
    class_name = str(video_row["class_name"])
    video_id = str(video_row["video_id"])
    video_path = str(video_row["video_path"])
    fps_original = float(video_row["fps_original"])
    frame_count = int(video_row["frame_count"])
    duration_seconds = float(video_row["duration_seconds"])

    if duration_seconds <= WINDOW_SECONDS:
        time_windows = [(0.0, round(duration_seconds, 6))]
    else:
        max_start = duration_seconds - WINDOW_SECONDS
        start_values = np.arange(0.0, max_start + 1e-9, STRIDE_SECONDS)
        time_windows = []
        for start_sec in start_values:
            time_windows.append(
                (
                    round(float(start_sec), 6),
                    round(float(start_sec + WINDOW_SECONDS), 6),
                )
            )

    for window_number, (start_sec, end_sec) in enumerate(time_windows):
        start_frame = int(round(start_sec * fps_original))
        start_frame = int(np.clip(start_frame, 0, max(frame_count - 1, 0)))

        end_frame = max(
            start_frame,
            min(frame_count - 1, math.ceil(end_sec * fps_original) - 1),
        )

        time_step = 1.0 / TARGET_FPS
        canonical_timestamps = start_sec + np.arange(0.0, max(end_sec - start_sec, 0.0), time_step)
        if canonical_timestamps.size == 0:
            canonical_timestamps = np.array([start_sec], dtype=float)

        sampled_positions = np.linspace(0, len(canonical_timestamps) - 1, num=FRAMES_PER_WINDOW)
        sampled_indices = np.round(sampled_positions).astype(int)
        sampled_timestamps = canonical_timestamps[sampled_indices]

        sampled_frame_indices = []
        rounded_timestamps = []
        for timestamp in sampled_timestamps:
            frame_index = int(round(float(timestamp) * fps_original))
            frame_index = int(np.clip(frame_index, 0, max(frame_count - 1, 0)))
            sampled_frame_indices.append(frame_index)
            rounded_timestamps.append(round(float(timestamp), 6))

        window_rows.append(
            {
                "split": split_name,
                "class_name": class_name,
                "video_id": video_id,
                "video_path": video_path,
                "window_id": f"{video_id}__w{window_number:04d}",
                "start_sec": round(float(start_sec), 6),
                "end_sec": round(float(end_sec), 6),
                "start_frame": int(start_frame),
                "end_frame": int(end_frame),
                "sampled_frame_indices": json.dumps(sampled_frame_indices),
                "sampled_timestamps": json.dumps(rounded_timestamps),
            }
        )

window_df = pd.DataFrame(window_rows).sort_values(["split", "class_name", "video_id", "window_id"]).reset_index(drop=True)

print(f"Ventanas generadas: {len(window_df)}")
display(
    window_df.groupby(["split", "class_name"])
    .agg(
        num_ventanas=("window_id", "size"),
        videos_cubiertos=("video_id", "nunique"),
        inicio_min_s=("start_sec", "min"),
        fin_max_s=("end_sec", "max"),
    )
    .round(2)
    .reset_index()
)


Ventanas generadas: 1885


,split,class_name,num_ventanas,videos_cubiertos,inicio_min_s,fin_max_s
0,test,hurto_simulado,93,17,0.0,20.0
1,test,normal,234,33,0.0,74.0
2,train,hurto_simulado,429,76,0.0,22.0
3,train,normal,855,153,0.0,24.0
4,val,hurto_simulado,89,16,0.0,20.0
5,val,normal,185,33,0.0,22.0


## Guardado de los manifests


In [5]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

video_manifest_path = OUTPUT_DIR / "video_manifest.csv"
window_manifest_path = OUTPUT_DIR / "window_manifest.csv"

video_df[
    [
        "split",
        "class_name",
        "video_path",
        "video_id",
        "fps_original",
        "frame_count",
        "duration_seconds",
        "width",
        "height",
    ]
].to_csv(video_manifest_path, index=False)

window_df[
    [
        "split",
        "class_name",
        "video_id",
        "video_path",
        "window_id",
        "start_sec",
        "end_sec",
        "start_frame",
        "end_frame",
        "sampled_frame_indices",
        "sampled_timestamps",
    ]
].to_csv(window_manifest_path, index=False)

print(video_manifest_path)
print(window_manifest_path)
print(f"Videos procesados: {len(video_df)}")
print(f"Ventanas generadas: {len(window_df)}")


c:\Users\franco\Downloads\1-INTELIGENCIA_ARTIFICIAL\PROJECT\ENTREGA_2\outputs\module1\video_manifest.csv
c:\Users\franco\Downloads\1-INTELIGENCIA_ARTIFICIAL\PROJECT\ENTREGA_2\outputs\module1\window_manifest.csv
Videos procesados: 328
Ventanas generadas: 1885


## Verificacion de artefactos


In [6]:
saved_video_manifest_df = pd.read_csv(video_manifest_path)
saved_window_manifest_df = pd.read_csv(window_manifest_path)

display(
    pd.DataFrame(
        [
            {
                "artifacto": "video_manifest.csv",
                "filas": len(saved_video_manifest_df),
                "columnas": saved_video_manifest_df.shape[1],
                "videos_unicos": saved_video_manifest_df["video_id"].nunique(),
            },
            {
                "artifacto": "window_manifest.csv",
                "filas": len(saved_window_manifest_df),
                "columnas": saved_window_manifest_df.shape[1],
                "videos_unicos": saved_window_manifest_df["video_id"].nunique(),
            },
        ]
    )
)

print("Videos por split y clase")
display(saved_video_manifest_df.groupby(["split", "class_name"]).size().reset_index(name="count"))

print("Ventanas por split y clase")
display(saved_window_manifest_df.groupby(["split", "class_name"]).size().reset_index(name="count"))

null_summary_df = pd.DataFrame(
    {
        "video_manifest_nulls": saved_video_manifest_df.isna().sum(),
        "window_manifest_nulls": saved_window_manifest_df.isna().sum(),
    }
).fillna("")

display(null_summary_df)


,artifacto,filas,columnas,videos_unicos
0,video_manifest.csv,328,9,328
1,window_manifest.csv,1885,11,328


Videos por split y clase


,split,class_name,count
0,test,hurto_simulado,17
1,test,normal,33
2,train,hurto_simulado,76
3,train,normal,153
4,val,hurto_simulado,16
5,val,normal,33


Ventanas por split y clase


,split,class_name,count
0,test,hurto_simulado,93
1,test,normal,234
2,train,hurto_simulado,429
3,train,normal,855
4,val,hurto_simulado,89
5,val,normal,185


,video_manifest_nulls,window_manifest_nulls
class_name,0.0,0.0
duration_seconds,0.0,
end_frame,,0.0
end_sec,,0.0
fps_original,0.0,
frame_count,0.0,
height,0.0,
sampled_frame_indices,,0.0
sampled_timestamps,,0.0
split,0.0,0.0
